# SAM Fine-tuning on Indian Face Data — AgeVision

**Use case:** Missing person — upload a 12-year-old boy's photo → age to 35 with accurate Indian face features.

**What this does:** Fine-tunes the `sam_ffhq_aging.pt` (2.27 GB) encoder on real Indian face images (UTKFace, ages 10-80). Decoder is frozen to preserve image quality.

## Prerequisites
1. Run `python pack_for_colab.py` locally → creates `colab_sam_training.zip`
2. Upload `colab_sam_training.zip` to Google Drive root (or any folder)
3. Open this notebook in Colab: **Runtime → Change runtime type → GPU (T4)**
4. Run all cells

In [ ]:
#@title Step 1: Mount Drive & Unpack Training Package
from google.colab import drive
drive.mount('/content/drive')

import os, sys, shutil, zipfile, re, csv, gc
import torch

WORK_DIR = '/content/agevision_sam'
DRIVE_DIR = '/content/drive/MyDrive'  # Root of Google Drive

# Find the zip file in Drive
ZIP_NAME = 'colab_sam_training.zip'
zip_path = None
for search_dir in [DRIVE_DIR, f'{DRIVE_DIR}/AgeVision', f'{DRIVE_DIR}/Colab']:
    candidate = os.path.join(search_dir, ZIP_NAME)
    if os.path.isfile(candidate):
        zip_path = candidate
        break

if zip_path is None:
    # Let user upload directly
    print(f'\n{ZIP_NAME} not found in Drive root.')
    print('Uploading directly...')
    from google.colab import files
    uploaded = files.upload()
    if ZIP_NAME in uploaded:
        zip_path = f'/content/{ZIP_NAME}'
        with open(zip_path, 'wb') as f:
            f.write(uploaded[ZIP_NAME])
    else:
        raise FileNotFoundError(
            f'Please upload {ZIP_NAME} to Google Drive or directly here'
        )

print(f'Found: {zip_path} ({os.path.getsize(zip_path)/1e9:.2f} GB)')

# Unpack
os.makedirs(WORK_DIR, exist_ok=True)
print('Extracting... (may take 2-3 minutes for 2.27 GB checkpoint)')
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(WORK_DIR)

# Set up paths
SAM_DIR = f'{WORK_DIR}/sam'
AGEVISION_API_DIR = f'{WORK_DIR}/agevision_api'
CHECKPOINTS_DIR = f'{WORK_DIR}/checkpoints'

# Create proper Python package structure
os.makedirs(AGEVISION_API_DIR, exist_ok=True)
sam_link = f'{AGEVISION_API_DIR}/sam'
if not os.path.exists(sam_link):
    os.symlink(SAM_DIR, sam_link)

# Create all __init__.py files
for d in [AGEVISION_API_DIR, SAM_DIR,
          f'{SAM_DIR}/configs', f'{SAM_DIR}/datasets', f'{SAM_DIR}/models',
          f'{SAM_DIR}/models/stylegan2', f'{SAM_DIR}/scripts', f'{SAM_DIR}/utils']:
    if os.path.isdir(d):
        init = os.path.join(d, '__init__.py')
        if not os.path.exists(init):
            open(init, 'w').close()

# Add to Python path
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

# CRITICAL: Write latest train_config.py to avoid version mismatch
train_config_path = f'{SAM_DIR}/train_config.py'
with open(train_config_path, 'w') as f:
    f.write('''\
"""
SAM Fine-tuning Configuration
"""
from dataclasses import dataclass

@dataclass
class SAMTrainConfig:
    checkpoint_path: str = ''
    output_dir: str = 'checkpoints/sam_finetune'
    data_root: str = ''
    age_csv: str = ''
    input_nc: int = 4
    output_size: int = 1024
    start_from_latent_avg: bool = False
    start_from_encoded_w_plus: bool = True
    batch_size: int = 2
    num_epochs: int = 20
    lr_encoder: float = 5e-5
    lr_decoder: float = 1e-5
    lr_discriminator: float = 1e-4
    weight_decay: float = 1e-5
    save_every: int = 5
    log_every: int = 10
    lambda_l1: float = 1.0
    lambda_identity: float = 1.0
    lambda_age: float = 0.1
    lambda_adv: float = 0.01
    lambda_lpips: float = 1.0
    img_size: int = 256
    num_workers: int = 2
    pin_memory: bool = True
    age_bins: int = 101
    freeze_decoder: bool = True
    gradient_accumulation_steps: int = 4
    gradient_checkpointing: bool = True
    device: str = 'cuda'
    mixed_precision: bool = True
''')
print('train_config.py written (latest version)')

# Use the original FFHQ model as base (clean, untouched)
CHECKPOINT_PATH = f'{CHECKPOINTS_DIR}/sam_ffhq_aging.pt'
print(f'\nSAM code: {SAM_DIR}')
print(f'Checkpoint: {CHECKPOINT_PATH} (exists: {os.path.isfile(CHECKPOINT_PATH)})')
print(f'Files: {os.listdir(SAM_DIR)}')

In [ ]:
#@title Step 2: Verify GPU
import torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu} ({mem:.1f} GB)')

    # SAM 1024px decoder needs batch=2 on T4, batch=4 on A100
    if mem >= 40:
        rec_batch = 4
    else:
        rec_batch = 2
    print(f'Recommended batch size: {rec_batch}')
else:
    raise RuntimeError(
        'No GPU! Go to Runtime -> Change runtime type -> GPU (T4 or better)'
    )

In [ ]:
#@title Step 3: Install Dependencies
!pip install -q gdown tqdm

In [ ]:
#@title Step 4: Download UTKFace & Filter Indian Faces (ages 10-80)
import glob

# Download UTKFace dataset
UTKFACE_DIR = f'{WORK_DIR}/utkface'
os.makedirs(UTKFACE_DIR, exist_ok=True)

# Try kagglehub first, fall back to gdown
if not glob.glob(f'{UTKFACE_DIR}/*.jpg'):
    try:
        import subprocess
        print('Downloading UTKFace dataset via kaggle...')
        subprocess.run(['pip', 'install', '-q', 'kaggle'], check=True)
        subprocess.run([
            'kaggle', 'datasets', 'download', '-d', 'jangedoo/utkface-new',
            '-p', UTKFACE_DIR, '--unzip'
        ], check=True)
    except Exception as e:
        print(f'Kaggle download failed ({e}), trying manual download...')
        print('Please upload UTKFace images manually to:', UTKFACE_DIR)
        print('Or download from: https://www.kaggle.com/datasets/jangedoo/utkface-new')
        raise FileNotFoundError(
            'Could not download UTKFace. Upload the images to Google Drive '
            'and update UTKFACE_DIR path, or install kaggle CLI.'
        )

# Set up dataset directories
DATASET_DIR = f'{WORK_DIR}/dataset'
IMAGES_DIR = f'{DATASET_DIR}/images'
os.makedirs(IMAGES_DIR, exist_ok=True)

MIN_AGE, MAX_AGE = 10, 80
pattern = re.compile(r'^(\d+)_(\d)_(\d)_(\d+)')

# Find all images in the UTKFace directory
all_images = []
for root, dirs, files in os.walk(UTKFACE_DIR):
    for fname in files:
        if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.chip.jpg')):
            all_images.append(os.path.join(root, fname))

print(f'Total UTKFace images found: {len(all_images)}')

if len(all_images) == 0:
    raise FileNotFoundError(
        f'No images found in {UTKFACE_DIR}. '
        'Download UTKFace dataset and place images there.'
    )

# Filter: race=3 (Indian), age 10-80
indian_samples = []
for img_path in all_images:
    fname = os.path.basename(img_path)
    match = pattern.match(fname)
    if not match:
        continue
    age = int(match.group(1))
    gender = int(match.group(2))  # 0=Male, 1=Female
    race = int(match.group(3))    # 3=Indian
    if race == 3 and MIN_AGE <= age <= MAX_AGE:
        indian_samples.append({
            'src': img_path, 'filename': fname,
            'age': age, 'gender': 'Male' if gender == 0 else 'Female'
        })

print(f'\nIndian faces (age {MIN_AGE}-{MAX_AGE}): {len(indian_samples)}')

if len(indian_samples) == 0:
    print('Sample files:')
    for p in all_images[:10]:
        print(f'  {os.path.basename(p)}')
    raise ValueError('No Indian faces (race=3) found in age range 10-80')

# Copy to dataset dir and create CSV
csv_path = f'{DATASET_DIR}/ages.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['filename', 'age', 'gender'])
    writer.writeheader()
    for s in indian_samples:
        dest = f'{IMAGES_DIR}/{s["filename"]}'
        if not os.path.exists(dest):
            shutil.copy2(s['src'], dest)
        writer.writerow({'filename': s['filename'], 'age': s['age'], 'gender': s['gender']})

# Age distribution
print('\nAge Distribution:')
age_dist = {}
for s in indian_samples:
    decade = (s['age'] // 10) * 10
    age_dist[decade] = age_dist.get(decade, 0) + 1
for d in sorted(age_dist):
    bar = '#' * (age_dist[d] // 3) or '|'
    print(f'  {d:2d}-{d+9:2d}: {age_dist[d]:4d}  {bar}')

males = sum(1 for s in indian_samples if s['gender'] == 'Male')
females = len(indian_samples) - males
print(f'\nMale: {males} | Female: {females}')
print(f'Dataset ready: {IMAGES_DIR} ({len(indian_samples)} images)')

In [ ]:
#@title Step 5: Configure Paths & Training

# Update paths_config for Colab environment
paths_config_path = f'{SAM_DIR}/configs/paths_config.py'
os.makedirs(os.path.dirname(paths_config_path), exist_ok=True)
for d in [f'{SAM_DIR}/configs', f'{SAM_DIR}/datasets', f'{SAM_DIR}/models',
          f'{SAM_DIR}/models/stylegan2', f'{SAM_DIR}/scripts', f'{SAM_DIR}/utils']:
    init = os.path.join(d, '__init__.py')
    if os.path.isdir(d) and not os.path.exists(init):
        open(init, 'w').close()

with open(paths_config_path, 'w') as f:
    f.write(f"""model_paths = {{
    'sam_ffhq_aging': '{CHECKPOINTS_DIR}/sam_ffhq_aging.pt',
    'sam_indian': '{CHECKPOINTS_DIR}/sam_indian_best.pt',
    'ir_se50': '{CHECKPOINTS_DIR}/model_ir_se50.pth',
    'stylegan_ffhq': '{CHECKPOINTS_DIR}/stylegan2-ffhq-config-f.pt',
    'pretrained_psp': '{CHECKPOINTS_DIR}/psp_ffhq_encode.pt',
    'age_predictor': '{CHECKPOINTS_DIR}/dex_age_classifier.pth',
}}\n""")
print('paths_config updated for Colab')

# Force-reload train_config to pick up the version written in Step 1
import importlib
import agevision_api.sam.train_config as _tc_mod
importlib.reload(_tc_mod)
from agevision_api.sam.train_config import SAMTrainConfig

# IMPORTANT: freeze_decoder=True keeps StyleGAN2 decoder frozen.
# Only the encoder is fine-tuned. This prevents quality degradation.
# batch_size=2 + accum=4 = effective batch 8 (fits T4 15GB VRAM)
config = SAMTrainConfig(
    data_root=IMAGES_DIR,
    age_csv=csv_path,
    checkpoint_path=CHECKPOINT_PATH,
    output_dir=f'{WORK_DIR}/checkpoints/sam_finetune',
    num_epochs=20,
    batch_size=2,
    lr_encoder=5e-5,
    lr_decoder=1e-5,
    lr_discriminator=1e-4,
    lambda_l1=1.0,
    lambda_identity=1.0,
    lambda_age=0.1,
    lambda_adv=0.01,
    lambda_lpips=1.0,
    device='cuda',
    mixed_precision=True,
    num_workers=2,
    save_every=5,
    log_every=10,
    freeze_decoder=True,
    gradient_checkpointing=True,
    gradient_accumulation_steps=4,
)

print(f'\nTraining config:')
print(f'  Dataset: {len(indian_samples)} Indian face images')
print(f'  Base checkpoint: sam_ffhq_aging.pt')
print(f'  Epochs: {config.num_epochs} | Batch: {config.batch_size} x {config.gradient_accumulation_steps} accum = {config.batch_size * config.gradient_accumulation_steps} effective')
print(f'  LR encoder: {config.lr_encoder}')
print(f'  freeze_decoder: {config.freeze_decoder}')
print(f'  lambda_identity: {config.lambda_identity} | lambda_lpips: {config.lambda_lpips}')
print(f'  Mixed precision: {config.mixed_precision}')
print(f'  Gradient checkpointing: {config.gradient_checkpointing}')
print(f'  Output: {config.output_dir}')

In [ ]:
#@title Step 6: START TRAINING
import logging
import importlib

# Fix: force logger output to show in Colab
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S',
    force=True,
)

# Reload train module to pick up any code changes from Step 1
import agevision_api.sam.train as _train_mod
importlib.reload(_train_mod)
from agevision_api.sam.train import train

print('='*60)
print('  SAM Fine-tuning on Indian Face Data')
print(f'  {len(indian_samples)} images | {config.num_epochs} epochs | GPU: {torch.cuda.get_device_name(0)}')
print(f'  Batch: {config.batch_size} x {config.gradient_accumulation_steps} accum = {config.batch_size * config.gradient_accumulation_steps} effective')
print(f'  freeze_decoder: {config.freeze_decoder}')
print('='*60)
print()

train(config)

print()
print('='*60)
print('  TRAINING COMPLETE!')
print(f'  Checkpoint: {WORK_DIR}/checkpoints/sam_finetune/sam_indian_finetuned.pt')
print('='*60)

In [ ]:
#@title Step 7: Evaluate — Before vs After (optional)
finetuned_path = f'{WORK_DIR}/checkpoints/sam_finetune/sam_indian_finetuned.pt'

if os.path.isfile(finetuned_path):
    try:
        from agevision_api.sam.evaluate import evaluate_checkpoint, compare_checkpoints
        print('Comparing original vs fine-tuned...')
        comparison = compare_checkpoints(
            before_path=CHECKPOINT_PATH,
            after_path=finetuned_path,
            data_root=IMAGES_DIR,
            age_csv=csv_path,
            device='cuda',
        )
        print(f'\n  BEFORE (sam_ffhq_aging.pt):')
        print(f'    L1 Error:       {comparison["before"]["avg_l1_error"]:.4f}')
        print(f'    Identity Score: {comparison["before"]["avg_identity_score"]:.4f}')
        print(f'\n  AFTER (fine-tuned):')
        print(f'    L1 Error:       {comparison["after"]["avg_l1_error"]:.4f}')
        print(f'    Identity Score: {comparison["after"]["avg_identity_score"]:.4f}')
    except Exception as e:
        print(f'Evaluation skipped: {e}')
        print('This is optional — your checkpoint is still saved.')
else:
    print(f'Checkpoint not found: {finetuned_path}')

In [ ]:
#@title Step 8: Visual Demo — Age a Young Face (optional)
try:
    from agevision_api.sam.evaluate import generate_comparison_grid
    from IPython.display import display, Image as IPImage

    # Find a young face (age 10-15) for the missing person demo
    young_faces = [s for s in indian_samples if 10 <= s['age'] <= 15]
    if young_faces:
        test_img = f"{IMAGES_DIR}/{young_faces[0]['filename']}"
        ages = [15, 20, 25, 30, 35, 40, 50, 60, 70]

        grid = generate_comparison_grid(
            checkpoint_path=finetuned_path,
            image_path=test_img,
            target_ages=ages,
            output_path=f'{WORK_DIR}/demo_age_progression.png',
            device='cuda',
        )
        print(f'Original face age: {young_faces[0]["age"]}')
        print(f'Progression to ages: {ages}')
        display(IPImage(filename=grid))
    else:
        print('No young faces (10-15) in dataset for demo')
except Exception as e:
    print(f'Demo skipped: {e}')
    print('This is optional — your checkpoint is still saved.')

In [ ]:
#@title Step 9: Save to Google Drive
DRIVE_SAVE_DIR = '/content/drive/MyDrive/AgeVision/checkpoints'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

finetuned_path = f'{WORK_DIR}/checkpoints/sam_finetune/sam_indian_finetuned.pt'

if os.path.isfile(finetuned_path):
    # Save as sam_indian_best.pt (for use in the app)
    dest = f'{DRIVE_SAVE_DIR}/sam_indian_best.pt'
    print(f'Saving to Drive: {dest}')
    shutil.copy2(finetuned_path, dest)
    size = os.path.getsize(dest) / 1e9
    print(f'Saved! ({size:.2f} GB)')

    # Timestamped backup
    from datetime import datetime
    ts = datetime.now().strftime('%Y%m%d_%H%M')
    backup = f'{DRIVE_SAVE_DIR}/sam_indian_finetuned_{ts}.pt'
    shutil.copy2(finetuned_path, backup)
    print(f'Backup: {backup}')

    print(f'\n--- NEXT STEPS ---')
    print(f'1. Download sam_indian_best.pt from Google Drive')
    print(f'   Path: AgeVision/checkpoints/sam_indian_best.pt')
    print(f'2. Copy to: agevision_backend/checkpoints/sam_indian_best.pt')
    print(f'3. Restart Django server and test')
else:
    print('No checkpoint to save')

In [ ]:
#@title Step 10: Download directly to your computer (optional)
from google.colab import files

finetuned_path = f'{WORK_DIR}/checkpoints/sam_finetune/sam_indian_finetuned.pt'

if os.path.isfile(finetuned_path):
    print(f'Starting download ({os.path.getsize(finetuned_path)/1e9:.2f} GB)...')
    files.download(finetuned_path)
else:
    print('No checkpoint to download. Run Step 6 (training) first.')